# Databricks: query-aware partition test

Create partition dates and filter values needed by SQL query-path tests in your own workspace.

In [ ]:
# Databricks notebook source
%pip install great-generator

In [ ]:
from great_generator import generate_from_schema, validate_query_coverage

schema = """
member_id string,
business_date date,
region string,
product_type string,
member_status string,
interaction_count int,
balance double
"""
required_values = {"region": ["SOUTH"], "product_type": ["CHECKING", "SAVINGS"], "member_status": ["ACTIVE"]}
partition_by = {"column": "business_date", "values": ["2026-01-01", "2026-01-02", "2026-01-03"], "distribution": "balanced"}

pdf = generate_from_schema(schema, rows=100000, required_values=required_values, partition_by=partition_by, seed=42)
report = validate_query_coverage(pdf, required_values=required_values, partition_by=partition_by)
report

In [ ]:
sdf = spark.createDataFrame(pdf)
path = "dbfs:/tmp/great-generator/query-aware-member-events"
sdf.write.format("delta").mode("overwrite").partitionBy("business_date").save(path)

spark.sql("DROP TABLE IF EXISTS default.synthetic_query_test")
spark.sql(f"CREATE TABLE default.synthetic_query_test USING DELTA LOCATION '{path}'")

In [ ]:
spark.sql('''
SELECT business_date, product_type, COUNT(*) AS rows
FROM default.synthetic_query_test
WHERE business_date BETWEEN '2026-01-01' AND '2026-01-03'
  AND region = 'SOUTH'
  AND product_type IN ('CHECKING', 'SAVINGS')
GROUP BY business_date, product_type
ORDER BY business_date, product_type
''').show()

This notebook tests query paths in your environment. It does not claim to reproduce production performance.